In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import altair as alt
from pathlib import Path
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# --- locate + load the databank ---
here = Path.cwd()
matches = list(here.rglob("PSF_aggregates_databank_Mar_EFO.xlsx"))
if not matches:
    for parent in here.parents:
        matches = list(parent.rglob("PSF_aggregates_databank_Mar_EFO.xlsx"))
        if matches: break
path = matches[0]

raw = pd.read_excel(path, sheet_name="Aggregates (per cent of GDP)", header=None)
d = raw.iloc[4:, [1, 11, 23, 32]].copy()
d.columns = ["year", "pb", "debt", "gap"]
d["year"] = d["year"].apply(lambda v: int(str(v)[:4]) if str(v)[:4].isdigit() else np.nan)
for c in ["pb", "debt", "gap"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
d = d.dropna(subset=["year", "pb", "debt"]).reset_index(drop=True)
d["year"] = d["year"].astype(int)
d = d[d["year"] <= 2024].sort_values("year").reset_index(drop=True)
d["debt_lag"] = d["debt"].shift(1)

# --- rolling 20-year beta ---
reg = d.dropna(subset=["debt_lag", "pb", "gap"]).reset_index(drop=True)
W = 20
rows = []
for i in range(len(reg) - W + 1):
    w = reg.iloc[i:i+W]
    m = smf.ols("pb ~ debt_lag + gap", data=w).fit()
    mid = int(w.year.iloc[W // 2])
    b, se = m.params["debt_lag"], m.bse["debt_lag"]
    rows.append((mid, b, b - 1.96*se, b + 1.96*se))
r = pd.DataFrame(rows, columns=["year", "beta", "lo", "hi"])
r["date"] = pd.to_datetime(r["year"], format="%Y")
r["series"] = "x"

# --- chart ---
zero = alt.Chart(pd.DataFrame({"y":[0]})).mark_rule(
    color="#122b39", strokeWidth=1).encode(y="y:Q")

band = alt.Chart(r).mark_area(opacity=0.15, color="#36B7B4").encode(
    x=alt.X("date:T", axis=alt.Axis(format="%Y", tickCount=7), title=None),
    y=alt.Y("lo:Q", title="Reaction coefficient (β on lagged debt)",
            scale=alt.Scale(domain=[-0.35, 0.35])),
    y2="hi:Q")

line = alt.Chart(r).mark_line(strokeWidth=2.4).encode(
    x="date:T", y="beta:Q", color=alt.Color("series:N", legend=None))

caption = alt.Title(
    text="Source: author's calculation; OBR public finances databank",
    subtitle=[
        "Rolling 20-year estimate of the UK fiscal reaction coefficient (primary balance on lagged debt).",
        "Above zero = debt-stabilising behaviour; below = not. Shaded band = 95% CI.",
    ],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (band + zero + line)
    .properties(width=700, height=320, title=caption)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, path="Charts", name="fiscal_reaction_rolling", svg=True)
chart.save("Charts/fiscal_reaction_rolling.png", scale_factor=2.0)
chart

alt.LayerChart(...)